# Instalar pacotes

In [2]:
!python3 -m pip install -r ../requirements.txt

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.0 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached MarkupSafe-3.0.2-cp313-cp313-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [folium]2m4/5 [folium]


In [10]:
from dotenv import dotenv_values
config = dotenv_values("../.env")

# 1 - Youtube Data API
## 1.1 Faça uma busca por 50 vídeos com a palavra chave 'eleição'.

In [11]:
from pyyoutube import Api

# get key from .env file
api = Api(api_key=config["YOUTUBE_APIKEY"])

In [12]:
videos_eleicao = api.search_by_keywords(q="eleição", count=50, search_type=["video"])

In [13]:
videos_eleicao.items

[SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#searchResult'),
 SearchResult(kind='youtube#sear

In [14]:
videos_eleicao.items[0].to_dict()

{'kind': 'youtube#searchResult',
 'etag': 'ozuXVX9dmYPaTvZNPAVMQnk_IsM',
 'id': {'kind': 'youtube#video',
  'videoId': 'z2qFZzAQB5s',
  'channelId': None,
  'playlistId': None},
 'snippet': {'publishedAt': '2025-08-20T16:54:37Z',
  'channelId': 'UC-ZkSRh-7UEuwXJQ9UMCFJA',
  'title': 'CPMI do INSS: reunião de instalação e eleição - 20/08/25 (parte 2)',
  'description': 'Parte 1: https://youtu.be/BS8n6sascs8 Comissão Parlamentar Mista de Inquérito do INSS - 2025 Tema: 1ª Reunião: instalação e ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/z2qFZzAQB5s/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/z2qFZzAQB5s/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/z2qFZzAQB5s/hqdefault.jpg',
    'width': 480,
    'height': 360},
   'standard': None,
   'maxres': None},
  'channelTitle': 'Câmara dos Deputados',
  'liveBroadcastContent': 'none'}}

## 1.2 Faça a procura por conteúdos em São Paulo no período do segundo turno da última eleição presidencial

In [15]:
videos_eleicoes_sp = api.search(location="-23.550487, -46.633125",
               location_radius="20mi",
               q="eleição",
               count=50,
               published_after="2022-10-03T00:00:00Z",
               published_before="2022-10-30T23:59:59Z",
               safe_search="moderate",
               search_type="video")

### 1.3 Identifique o vídeo com mais comentários e exiba as informações desse vídeo

In [16]:
video_mais_comentarios = api.get_video_by_id(video_id=videos_eleicoes_sp.items[0].id.videoId)
maximo_comentarios = int(video_mais_comentarios.items[0].statistics.commentCount) if video_mais_comentarios.items[0].statistics.commentCount is not None else 0

for video in videos_eleicoes_sp.items[1:]:
  video_id = video.id.videoId
  video_details = api.get_video_by_id(video_id=video_id)
  contagem_comentarios = video_details.items[0].statistics.commentCount
  cont = int(contagem_comentarios) if contagem_comentarios is not None else 0
  if cont > maximo_comentarios:
    video_mais_comentarios = video_details
    maximo_comentarios = cont

video_mais_comentarios.to_dict()

{'kind': 'youtube#videoListResponse',
 'etag': 'xIXEBoYDN-16MEITvrQUVa0nH4k',
 'nextPageToken': None,
 'prevPageToken': None,
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 1},
 'items': [{'kind': 'youtube#video',
   'etag': 'uROYIwsOF0hp62hJFDLZeplaiYU',
   'id': 'b2tkImEjBHY',
   'snippet': {'publishedAt': '2022-10-03T02:30:47Z',
    'channelId': 'UCcw5LxOx4RABRj4h3M15NcA',
    'title': 'A educação venceu!',
    'description': 'Cada um de vocês fez isso acontecer. A luta em defesa da educação vai seguir na Alesp, com o professor Carlos Giannazi. Também tivemos uma grande vitória com a professora Luciene Cavalcante como primeira suplente, em uma eleição tão difícil.\n\nA nossa luta continua! Vamos juntos.\n\n\n\n\nINSCREVA-SE  NO CANAL  DO DEPUTADO PROFESSOR CARLOS GIANNAZI\nhttps://www.youtube.com/user/CarlosGiannazi?sub_confirmation=1\n---\n// REDES SOCIAIS\nfacebook.com/carlosgiannazioficial\ntwitter.com/carlosgiannazi\ninstagram.com/carlosgiannazioficial\n\n// VISITE O SITE \n

# 2 - Faça um coletor que capture as reviews da página principal de:
https://www.apontador.com.br/local/sp/campinas/faculdades_e_universidades/88Q2E372/unicamp.html.

In [17]:
from bs4 import BeautifulSoup
import requests

url="https://www.apontador.com.br/local/sp/campinas/faculdades_e_universidades/88Q2E372/"

req = requests.get(url)
soup = BeautifulSoup(req.content,'html.parser')

In [18]:
avaliacoes = soup.find_all("div", class_="store-comments__item")

for avaliacao in avaliacoes:
  nome = avaliacao.find("span", class_="name").text
  data = avaliacao.find("span", class_="rate-stars__amount").text
  texto = avaliacao.find("div", class_="store-comments__text").find("p").text
  print(f"Nome: {nome} - {data} - Comentário: {texto}\n")


Nome: Anderson Thees - publicada em 19/08/2010 - Comentário: A UNICAMP é uma universidade muito boa. O fato de estar um pouco distante de Campinas (40 mins) promove uma experiência diferenciada, criando uma comunidade muito mais engajada. A qualidade do ensino é de padrão mundial. Orgulho de ser ex-aluno!

Nome: Eduardo M. Maçan - publicada em 12/09/2010 - Comentário: Cursei engenharia de computação na Unicamp e foram os melhores anos da minha vida. O ambiente universitário, a abundância de conhecimento ao meu redor em todas as áreas, tudo isso contribuiu para minha formação, não apenas técnica.

Voltaria a qualquer momento e a recomendo para quem quiser usufruir do melhor ensino do país, estando apto a pagar com a dedicação necessária.

Nome: Thiago Ganzarolli Da Silva - publicada em 25/11/2011 - Comentário: É muito agradável estudar em um campus grande, com muitas locais para estudar: mesas no ciclo básico, biblioteca central ou de outros institutos, uma sala vazia...
Uma universidad

# 3 - Crie uma aplicação que usa o Selenium para abrir o site https://www.uol.com.br/ e rolar a página para baixo até o fim do conteúdo.

In [19]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

options = Options()

driver = webdriver.Chrome(options=options)
driver.get("https://www.uol.com.br/")

last_height = driver.execute_script("return document.body.scrollHeight")

while True:
  driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
  time.sleep(2)
  new_height = driver.execute_script("return document.body.scrollHeight")
  if new_height == last_height:
    break
  last_height = new_height

driver.quit()